# Check prepared correction inputs

After these checks, run central correction from the repository root:

```bash
pixi run python -m evaluation_utils.run_correction_experiments correct --datasets quartet_multiomics
```


In [7]:
library(tidyverse)

In [8]:
library(tidyverse)
source(file.path(if (dir.exists("figshare_data")) "." else "evaluation_data/quartet_multiomics",
                 "..", "..", "evaluation_utils", "plots_eda.R"))

WORKDIR <- if (dir.exists("figshare_data")) "." else "evaluation_data/quartet_multiomics"
BEFORE_DIR <- file.path(WORKDIR, "before")
PLOTS_DIR <- file.path(WORKDIR, "plots")

dir.create(PLOTS_DIR, showWarnings = FALSE, recursive = TRUE)

MODALITIES <- c("Transcriptomics", "Proteomics", "Metabolomics")
# Reference donor first; D6 becomes the limma intercept condition.
DONOR_LEVELS <- c("D6", "D5", "F7", "M8")
DONOR_REFERENCE <- DONOR_LEVELS[1]
# Toggle: include the synthetic client_04_L03_L14 (L03 + L14 fold-in).
# Default FALSE -- must match the toggles in 01_preprocess_eda.ipynb and
# 02_prepare_RBE_inputs.ipynb. Used here only as documentation; correction
# operates on whichever clients are present in before/<Modality>/metadata.tsv.
INCLUDE_CLIENT_04 <- FALSE
ALL_CLIENT_NAMES <- c("client_01_L01", "client_02_L02",
                     "client_03_L05_L04", "client_04_L03_L14")
OPTIONAL_CLIENTS <- c("client_04_L03_L14")
CLIENT_NAMES <- if (INCLUDE_CLIENT_04) {
  ALL_CLIENT_NAMES
} else {
  setdiff(ALL_CLIENT_NAMES, OPTIONAL_CLIENTS)
}
N_DONORS <- length(DONOR_LEVELS)


## Helper functions

`plot_diagnostics` uses `pca_plot()`, `boxplot_plot()`, and `plotIntensityDensity()` from `evaluation_utils/plots_eda.R`, combined into one patchwork panel per (modality, state).


In [9]:
read_feature_matrix <- function(path) {
  df <- readr::read_tsv(path, show_col_types = FALSE)
  first_col <- names(df)[1]
  mat <- df %>%
    column_to_rownames(first_col) %>%
    as.data.frame(check.names = FALSE)
  mat[] <- lapply(mat, as.numeric)
  as.matrix(mat)
}

load_prepared_modality <- function(modality) {
  modality_dir <- file.path(BEFORE_DIR, modality)
  expr <- read_feature_matrix(file.path(modality_dir, "central_intensities_log_UNION.tsv"))
  metadata <- readr::read_tsv(file.path(modality_dir, "metadata.tsv"), show_col_types = FALSE) %>%
    mutate(
      condition = factor(condition, levels = DONOR_LEVELS),
      batch = factor(batch),
      rep = as.integer(rep)
    )

  missing <- setdiff(metadata$file, colnames(expr))
  if (length(missing) > 0) stop(modality, ": missing matrix samples: ", paste(head(missing), collapse = ", "), call. = FALSE)
  expr <- expr[, metadata$file, drop = FALSE]
  list(expr = expr, metadata = metadata)
}

# Validates balanced donor/batch structure and full design-matrix rank.
assert_limma_design <- function(metadata, modality) {
  donor_counts <- metadata %>% count(condition) %>% pull(n)
  if (n_distinct(donor_counts) != 1) {
    stop(modality, ": unbalanced donors (", paste(donor_counts, collapse = "/"), ").", call. = FALSE)
  }
  cell_counts <- metadata %>% count(batch_code, batch, condition) %>% pull(n)
  if (n_distinct(cell_counts) != 1) {
    stop(modality, ": unequal replicates per (batch, condition) cell.", call. = FALSE)
  }

  full_design <- model.matrix(~ condition + batch, data = metadata)
  rank <- qr(full_design)$rank
  if (rank != ncol(full_design)) {
    stop(modality, ": limma design is not full rank: ", rank, " < ", ncol(full_design), call. = FALSE)
  }
  invisible(TRUE)
}


## Check each modality independently


In [10]:
diagnostic_summaries <- list()

preparation_summaries <- purrr::map_dfr(MODALITIES, function(modality) {
  message("Checking ", modality)
  prepared <- load_prepared_modality(modality)
  expr <- prepared$expr
  metadata <- prepared$metadata
  assert_limma_design(metadata, modality)

  diagnostic_summaries[[paste(modality, "before", sep = "_")]] <<-
    plot_diagnostics(expr, metadata, modality, "before", PLOTS_DIR)
  file.copy(file.path(BEFORE_DIR, modality, "central_intensities_log_UNION.tsv"),
            file.path(BEFORE_DIR, modality, "central_input.tsv"), overwrite = TRUE)

  tibble(
    modality = modality,
    features = nrow(expr),
    samples = ncol(expr),
    input_missing_cells = sum(is.na(expr)),
    rows_with_any_na = sum(rowSums(is.na(expr)) > 0),
    batches = n_distinct(metadata$batch),
    design_rank = qr(model.matrix(~ condition + batch, data = metadata))$rank,
    design_columns = ncol(model.matrix(~ condition + batch, data = metadata))
  )
})

preparation_summaries


Checking Transcriptomics

Checking Proteomics

Checking Metabolomics



modality,features,samples,input_missing_cells,rows_with_any_na,batches,design_rank,design_columns
<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
Transcriptomics,26907,48,0,0,4,7,7
Proteomics,2539,48,0,0,4,7,7
Metabolomics,71,48,0,0,4,7,7


## Preparation summaries


In [11]:
diagnostic_summaries_df <- bind_rows(diagnostic_summaries)

preparation_summaries
diagnostic_summaries_df


modality,features,samples,input_missing_cells,rows_with_any_na,batches,design_rank,design_columns
<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
Transcriptomics,26907,48,0,0,4,7,7
Proteomics,2539,48,0,0,4,7,7
Metabolomics,71,48,0,0,4,7,7


modality,state,pca_features
<chr>,<chr>,<int>
Transcriptomics,before,26907
Proteomics,before,2539
Metabolomics,before,71


## Session information

In [12]:
sessionInfo()

R version 4.5.3 (2026-03-11)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 26.04 LTS

Matrix products: default
BLAS/LAPACK: /home/yuliya/repos/cosybio/fedRBE/.pixi/envs/default/lib/libopenblasp-r0.3.34.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Europe/Berlin
tzcode source: system (glibc)

attached base packages:
[1] grid      stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] viridis_0.6.5     viridisLite_0.4.3 ggsci_5.2.0       umap_0.2.10.0    
 [5] patchwork_1.3.2   gridExtra_2.3.1   lubridate_1.9.5   forcats_1.0.1    
 [9] stringr_1.6.0     dplyr_1.2.1       purrr_1.2